In [1]:
import sys
sys.path.append("..")

import os
import pandas as pd
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from pprint import pp
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.utils.visualize import mask_to_alpha_image

In [2]:
output_dir = "../outputs"
run_groups = os.listdir(output_dir)
run_groups.sort()

def update_run_selector(change):
    runs = os.listdir(os.path.join(output_dir, change['new']))
    runs.sort()
    run_selector.options = runs
    run_selector.value = runs
    run_selector.layout.height = f"{8+len(runs)*20}px"

run_group_selector = widgets.ToggleButtons(options=run_groups, description="Select run group:")
run_group_selector.observe(update_run_selector, names="value")

runs = os.listdir(f"../outputs/{run_group_selector.value}")
runs.sort()
run_selector = widgets.SelectMultiple(
    options=runs,
    value=runs,
    description='Select runs:',
)
run_selector.layout.height = f"{8+len(runs)*19}px"

display(run_group_selector, run_selector)

ToggleButtons(description='Select run group:', options=('2026-03-12', 'code-test', 'code-test2', 'fourier_feat…

SelectMultiple(description='Select runs:', index=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17…

In [3]:
def flatten_config(conf):
    dict_ = OmegaConf.to_container(conf, resolve=True)
    return flatten_config_helper(dict_)

def flatten_config_helper(original, prefix=""):
    result = {}
    for key, val in original.items():
        if isinstance(val, dict):
            result = result | flatten_config_helper(val, prefix + key + ".")
        elif isinstance(val, list):
            if len(val)>0 and isinstance(val[0], dict): # flatten dicts to make hashable
                for dict_ in val:
                    result = result | flatten_config_helper(dict_, prefix + key + ".")
            else: # convert to tuple to make it hashable
                result[prefix + key] = tuple(val)
        else:
            result[prefix + key] = val
    return result

def config_diff(configs):
    sets = [set(conf.items()) for conf in configs]
    first_set = sets[0]
    set_diff = set()
    for set_ in sets[1:]:
        set_diff = set_diff.union(first_set ^ set_)
    return list(dict(set_diff).keys())

runs = run_selector.value
run_paths = [os.path.join(output_dir, run_group_selector.value, run) for run in runs]

config_list = [OmegaConf.load(os.path.join(path, ".hydra/config.yaml")) for path in run_paths]
for i, conf in enumerate(config_list):
    conf.run_name = runs[i]

if len(config_list) > 1:
    flattened_configs = [flatten_config(conf) for conf in config_list]
    config_diff_keys = config_diff(flattened_configs)
    conf_selector = widgets.SelectMultiple(
        options=config_diff_keys,
        value=config_diff_keys,
        description='Select config parameters to include:',
        )
    conf_selector.layout.height = f"{8+len(config_diff_keys)*19}px"
    display(conf_selector)

SelectMultiple(description='Select config parameters to include:', index=(0, 1, 2), layout=Layout(height='65px…

In [4]:
if len(config_list) > 1:
    constant_config = {key: val for key, val in flattened_configs[0].items() if key not in config_diff_keys}
    selected_params = [{key: val for key, val in conf.items() if key in conf_selector.value} for conf in flattened_configs]
    non_selected_params = [{key: val for key, val in conf.items() if key in config_diff_keys and key not in conf_selector.value} for conf in flattened_configs]
    results_list = [pd.read_pickle(os.path.join(path, "results.pickle")) for path in run_paths]
    for i, results in enumerate(results_list):
        for name, val in selected_params[i].items():
            results[name] = val
    all_results = pd.concat(results_list)
    print("Shared config:")
    pp(constant_config)
    print("\nSelected parameters:")
    pp(selected_params)
    print("\nNon-selected parameters:")
    pp(non_selected_params)
else:
    all_results = pd.read_pickle(os.path.join(run_paths[0], "results.pickle"))
    print(OmegaConf.to_yaml(config_list[0]))

Shared config:
{'model._target_': 'src.models.RandomFourierFeatures',
 'model.in_features': 2,
 'model.fourier_features': 256,
 'model.hidden_features': 256,
 'model.hidden_layers': 3,
 'model.out_features': 1,
 'optimizer._target_': 'torch.optim.Adam',
 'optimizer._partial_': True,
 'optimizer.lr': 0.0005326,
 'run_group': 'testerino7',
 'epochs': 750,
 'save_images': True,
 'save_running_metrics': True,
 'criterion._target_': 'torch.nn.MSELoss',
 'metrics.PSNR._target_': 'src.utils.metrics.PSNR',
 'metrics.PSNR.data_range': 1.0,
 'metrics.RMSE._target_': 'src.utils.metrics.RMSE',
 'metrics.MAE._target_': 'torch.nn.L1Loss',
 'metrics.ME._target_': 'src.utils.metrics.ME',
 'metrics.R2._target_': 'src.utils.metrics.R2',
 'data.opencanopy_indices': (262, 819),
 'data.normalize_image': True,
 'data.test_grid_skip': 2,
 'dataloader.batch_size': 'full',
 'dataloader.shuffle': True}

Selected parameters:
[{'model.scale': 1, 'data.sparsity': 0.5},
 {'model.scale': 1, 'data.sparsity': 0.2},
 {

In [5]:
#######################################################################
# Source - https://stackoverflow.com/a/34162576
# Posted by DSM
# Retrieved 2026-04-05, License - CC BY-SA 3.0

def query_df(df, dict_):
    return df.loc[(df[list(dict_)] == pd.Series(dict_)).all(axis=1)]

#######################################################################

all_results = all_results.fillna("nan")

all_metrics = ["loss", "PSNR", "RMSE", "MAE", "ME", "R2"]
available_metrics = [x for x in all_metrics if any(x in column_name for column_name in all_results.columns)]

options_dict = {key: all_results[key].unique() for key in conf_selector.value}

def plot_metric(query_dict, metric, log_scale):
    fig, ax = plt.subplots()
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        print("Not unique combination!")
    elif len(query_result) == 0:
        print("No run satisfies combination!")
    else:
        ax.plot(query_result[f"all_train_{metric}"].item(), label=f"train_{metric}")
        ax.plot(query_result[f"all_test_{metric}"].item(), label=f"test_{metric}")
        if log_scale:
            ax.set_yscale("log")
        ax.legend()
        display(fig)

def plot_images(query_dict, cmap="summer"):
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        print("Not unique combination!")
    elif len(query_result) == 0:
        print("No run satisfies combination!")
    else:
        fig, ax = plt.subplots(3, 2, figsize=(12,18))
        for i in range(3):
            ax[i, 0].imshow(query_result["target_image"].item(), cmap=cmap)
            ax[i, 0].set_title("target image")
        
        ax[0, 1].imshow(query_result["target_image"].item(), cmap=cmap)
        mask = query_result["train_mask"].item()
        ax[0, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[0, 1].set_title("sparse (train)")
    
        ax[1, 1].imshow(query_result["reconstructed_target"].item(), cmap=cmap)
        ax[1, 1].set_title("reconstructed")
    
        ax[2, 1].imshow(query_result["target_image"].item(), cmap=cmap)
        mask = query_result["test_mask"].item()
        ax[2, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[2, 1].set_title("test")
        for something in ax:
            for this in something:
                this.set_axis_off()
        display(fig)

def make_plots(change=None):
    with out:
        out.clear_output(wait=True)
        chosen_options = {list(options_dict)[i]: sel.value for i, sel in enumerate(options_selectors)}
        chosen_options["opencanopy_idx"] = opencanopy_idx_selector.value
        plot_metric(query_dict=chosen_options, metric=metric_selector.value, log_scale=log_scale_selector.value)
        plot_images(query_dict=chosen_options)


options_selectors = [widgets.ToggleButtons(options=vals, description=key+":") for key, vals in options_dict.items()]
opencanopy_idx_selector = widgets.Dropdown(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
metric_selector = widgets.Dropdown(
    options=available_metrics,
    value='loss',
    description='metric:',
)
log_scale_selector = widgets.Checkbox(
    value=False,
    description='log scale:',
)

for selector in options_selectors:
    selector.observe(make_plots, names="value")
metric_selector.observe(make_plots, names="value")
log_scale_selector.observe(make_plots, names="value")
opencanopy_idx_selector.observe(make_plots, names="value")


for selector in options_selectors:
    display(selector)
print("--------------------------------------------------------------------------------")
display(widgets.HBox([metric_selector, log_scale_selector]))
display(opencanopy_idx_selector)

out = widgets.Output()
out

ToggleButtons(description='data.sparsity:', options=(np.float64(0.5), np.float64(0.2), np.float64(0.1), np.flo…

ToggleButtons(description='model.scale:', options=(np.int64(1), np.int64(2), np.int64(4), np.int64(6), np.int6…

--------------------------------------------------------------------------------


Dropdown(description='opencanopy idx:', options=(np.int64(262), np.int64(819)), value=np.int64(262))

Output()

In [6]:
metric_columns = [f"train_{metric}" for metric in available_metrics] + [f"test_{metric}" for metric in available_metrics]
relevant_columns = list(conf_selector.value) + ["opencanopy_idx"] + metric_columns
final_metrics = all_results[relevant_columns]
final_metrics

,data.sparsity,model.scale,opencanopy_idx,train_loss,train_PSNR,train_RMSE,train_MAE,train_ME,train_R2,test_loss,test_PSNR,test_RMSE,test_MAE,test_ME,test_R2
0,0.50,1,262,tensor(0.0046),tensor(23.3946),tensor(0.0677),tensor(0.0403),tensor(-0.0044),tensor(0.8698),tensor(0.0052),tensor(22.8668),tensor(0.0719),tensor(0.0420),tensor(-0.0022),tensor(0.8547)
1,0.50,1,819,tensor(0.0022),tensor(26.5820),tensor(0.0469),tensor(0.0194),tensor(-0.0009),tensor(0.9503),tensor(0.0024),tensor(26.1819),tensor(0.0491),tensor(0.0200),tensor(-0.0030),tensor(0.9457)
0,0.20,1,262,tensor(0.0044),tensor(23.5756),tensor(0.0663),tensor(0.0396),tensor(-0.0140),tensor(0.8776),tensor(0.0063),tensor(22.0137),tensor(0.0793),tensor(0.0462),tensor(0.0052),tensor(0.8232)
1,0.20,1,819,tensor(0.0019),tensor(27.2978),tensor(0.0432),tensor(0.0188),tensor(-0.0060),tensor(0.9578),tensor(0.0027),tensor(25.7480),tensor(0.0516),tensor(0.0212),tensor(0.0020),tensor(0.9400)
0,0.10,1,262,tensor(0.0040),tensor(23.9327),tensor(0.0636),tensor(0.0385),tensor(-0.0098),tensor(0.8848),tensor(0.0075),tensor(21.2436),tensor(0.0867),tensor(0.0499),tensor(-0.0037),tensor(0.7888)
1,0.10,1,819,tensor(0.0016),tensor(27.8554),tensor(0.0405),tensor(0.0173),tensor(0.0033),tensor(0.9632),tensor(0.0031),tensor(25.0350),tensor(0.0560),tensor(0.0229),tensor(-0.0056),tensor(0.9293)
0,0.01,1,262,tensor(0.0004),tensor(34.2038),tensor(0.0195),tensor(0.0122),tensor(-0.0032),tensor(0.9894),tensor(0.0171),tensor(17.6817),tensor(0.1306),tensor(0.0764),tensor(-0.0134),tensor(0.5205)
1,0.01,1,819,tensor(0.0004),tensor(34.4740),tensor(0.0189),tensor(0.0068),tensor(-0.0044),tensor(0.9922),tensor(0.0115),tensor(19.3908),tensor(0.1073),tensor(0.0446),tensor(-0.0025),tensor(0.7405)
0,0.50,2,262,tensor(0.0030),tensor(25.2610),tensor(0.0546),tensor(0.0294),tensor(-0.0044),tensor(0.9164),tensor(0.0044),tensor(23.5442),tensor(0.0665),tensor(0.0350),tensor(-0.0031),tensor(0.8757)
1,0.50,2,819,tensor(0.0009),tensor(30.5053),tensor(0.0298),tensor(0.0133),tensor(-0.0004),tensor(0.9801),tensor(0.0014),tensor(28.6532),tensor(0.0369),tensor(0.0154),tensor(-0.0005),tensor(0.9693)


In [7]:
final_metrics.groupby(["data.sparsity", "model.scale"]).mean().sort_values(["data.sparsity", "test_loss"], ascending=True)

opencanopy_idx train_loss train_PSNR train_RMSE  \
data.sparsity model.scale                                                    
0.01          1                     540.5   0.000368  34.338902   0.019191   
              2                     540.5   0.000089    40.6343   0.009369   
              4                     540.5   0.000007  51.827255   0.002595   
              6                     540.5   0.000003  54.697029   0.001842   
              8                     540.5    0.00001  53.390884   0.002669   
0.10          1                     540.5   0.002841  25.894073   0.052033   
              2                     540.5   0.000634  32.357708    0.02465   
              4                     540.5   0.000174  37.742943   0.013076   
              6                     540.5   0.000091  40.428947   0.009519   
              8                     540.5   0.000019  47.493847   0.004285   
0.20          2                     540.5   0.001105  30.474295   0.031637   
              1                     540.5   0.003126  25.436722   0.054709   
              4                     540.5   0.000307  35.511108   0.017151   
              6                     540.5   0.000114  39.652466   0.010545   
              8                     540.5   0.000068   41.74353   0.008226   
0.50          4                     540.5   0.000758  32.073051   0.026251   
              2                     540.5   0.001934  27.883158   0.042203   
              6                     540.5   0.000397  34.413177   0.019477   
              8                     540.5    0.00023   36.57143    0.01501   
              1                     540.5   0.003387  24.988274   0.057261   

                          train_MAE  train_ME  train_R2 test_loss  test_PSNR  \
data.sparsity model.scale                                                      
0.01          1            0.009519 -0.003817   0.99077   0.01428   18.53623   
              2            0.002788 -0.000828  0.997914  0.017875  17.749125   
              4             0.00092  0.000149  0.999826  0.020544  16.880009   
              6            0.000713   0.00017  0.999916  0.029112  15.362023   
              8            0.000834  0.000299  0.999788  0.036395  14.398323   
0.10          1            0.027911  -0.00324  0.923991  0.005323  23.139301   
              2             0.01292  0.000594  0.983189  0.005721  22.888073   
              4            0.006493  0.000387  0.995445  0.007123  21.984489   
              6            0.003232 -0.000574  0.997665   0.00772  21.613319   
              8            0.001982  0.000244  0.999504  0.009023  20.728077   
0.20          2            0.016283 -0.001782  0.970159  0.003874  24.746038   
              1            0.029197 -0.009972  0.917718  0.004476  23.880892   
              4            0.008613 -0.001777   0.99182  0.004708  24.006847   
              6            0.004968 -0.000575  0.997024    0.0052  23.416027   
              8            0.004104 -0.000792  0.998244  0.005594  23.183174   
0.50          4            0.013428 -0.002426  0.979559  0.002861  26.404949   
              2            0.021327 -0.002373  0.948244  0.002893  26.098682   
              6             0.00994 -0.003503  0.989453  0.003086  26.031557   
              8            0.008034 -0.002732  0.993991  0.003117  26.057888   
              1            0.029849 -0.002666  0.910044  0.003788  24.524321   

                          test_RMSE  test_MAE   test_ME   test_R2  
data.sparsity model.scale                                          
0.01          1            0.118929  0.060475  -0.00794  0.630517  
              2            0.131656  0.067548 -0.003428  0.530096  
              4            0.143275  0.077919 -0.033579  0.476336  
              6            0.170596  0.098598 -0.048833  0.265392  
              8            0.190679  0.113895 -0.063821  0.084416  
0.10          1            0.071334  0.036405 -0.004642  0.859054  
     

In [8]:
# let us assume all combinations of chosen parameters are present
def make_image_grid(opencanopy_idx, x_var, y_var, flip_x, flip_y, image_name, mask_name, metric, log_scale, color_within):
    assert (image_name is None and metric is not None) or (image_name is not None and metric is None)
    size = 4
    x_vals = all_results[x_var].sort_values(ascending=flip_x).unique()
    y_vals = all_results[y_var].sort_values(ascending=flip_y).unique()
    x_len = len(x_vals)
    y_len = len(y_vals)
    fig, ax = plt.subplots(y_len, x_len, figsize=(size*x_len, size*y_len))
    for i, y_val in enumerate(y_vals):
        if color_within == y_var:
            min_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val}")[f"test_loss"].min()
            max_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val}")[f"test_loss"].max()
        for j, x_val in enumerate(x_vals):
            if color_within == x_var:
                min_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{x_var}` == {x_val}")[f"test_loss"].min()
                max_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{x_var}` == {x_val}")[f"test_loss"].max()
            if metric:
                ax[i, j].plot(all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[f"all_train_{metric}"].item(), label=f"train_{metric}")
                ax[i, j].plot(all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[f"all_test_{metric}"].item(), label=f"test_{metric}")
                if log_scale:
                    ax[i, j].set_yscale("log")
                ax[i, j].legend()
            else:
                ax[i, j].imshow(all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[image_name].item(), cmap="summer")
                if mask_name is not None:
                    mask = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[mask_name].item()
                    ax[i, j].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
                ax[i, j].set_xticks([], [])
                ax[i, j].set_yticks([], [])
                loss_value = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[f"test_loss"].item()
                ax[i, j].patch.set_edgecolor((float((max_loss-loss_value)/(max_loss-min_loss))**1, 0.0, 0.0))
                ax[i, j].patch.set_linewidth(8)
            ax[i, j].set_title(f"{y_var}={y_val}, {x_var}={x_val}")
    fig.tight_layout()
    display(fig)

def update_image_grid(update):
    match image_selector.value:
        case "target":
            image_column = "target_image"
            mask_column = None
            metric = None
        case "sparse":
            image_column = "target_image"
            mask_column = "train_mask"
            metric = None
        case "reconstructed":
            image_column = "reconstructed_target"
            mask_column = None
            metric = None
        case "test_grid":
            image_column = "target_image"
            mask_column = "test_mask"
            metric = None
        case "metric":
            image_column = None
            mask_column = None
            metric = metric_selector_grid.value
    with out2:
        out2.clear_output(wait=True)
        make_image_grid(opencanopy_idx_selector_grid.value, x_var_selector.value, y_var_selector.value, x_flip.value, y_flip.value, image_column, mask_column, metric, log_scale_selector_grid.value, color_within_selector.value)

opencanopy_idx_selector_grid = widgets.ToggleButtons(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
image_selector = widgets.ToggleButtons(
    options=["target", "sparse", "reconstructed", "test_grid", "metric"],
    description="show:"
)
metric_selector_grid = widgets.Dropdown(
    options=available_metrics,
    description="metric:"
)
log_scale_selector_grid = widgets.Checkbox(
    description="log scale"
)
x_var_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="x_var:"
)
y_var_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="y_var:"
)
x_flip = widgets.Checkbox(
    description="flip x-axis:"
)
y_flip = widgets.Checkbox(
    description="flip y-axis:"
)
color_within_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="color within:"
)

opencanopy_idx_selector_grid.observe(update_image_grid, names="value")
image_selector.observe(update_image_grid, names="value")
x_var_selector.observe(update_image_grid, names="value")
y_var_selector.observe(update_image_grid, names="value")
x_flip.observe(update_image_grid, names="value")
y_flip.observe(update_image_grid, names="value")
metric_selector_grid.observe(update_image_grid, names="value")
log_scale_selector_grid.observe(update_image_grid, names="value")
color_within_selector.observe(update_image_grid, names="value")

display(opencanopy_idx_selector_grid)
display(widgets.HBox([x_var_selector, x_flip]))
display(widgets.HBox([y_var_selector, y_flip]))
display(widgets.HBox([image_selector, metric_selector_grid, log_scale_selector_grid]))
display(color_within_selector)

out2 = widgets.Output()
out2

ToggleButtons(description='opencanopy idx:', options=(np.int64(262), np.int64(819)), value=np.int64(262))

ToggleButtons(description='color within:', options=('data.sparsity', 'model.scale'), value='data.sparsity')

Output()

In [ ]:
# Beware
# Either give range or provide colorbar for images. Maybe its good to give range either ways and maybe also for metrics.